In [2]:
import pandas as pd

cargoterminal_el_demand = pd.read_excel("inputs/aligned_cargoterminal_el_demand.xlsx")

In [3]:
# Take first two columns, exclude first data row after header,
# rename columns, and parse datetime
cargoterminal_el_demand = (
    cargoterminal_el_demand.iloc[1:, [0, 1]]
      .copy()
      .rename(columns={cargoterminal_el_demand.columns[0]: "datetime", cargoterminal_el_demand.columns[1]: "ampere"})
)
cargoterminal_el_demand["datetime"] = pd.to_datetime(
    cargoterminal_el_demand["datetime"], errors="coerce"
)
cargoterminal_el_demand["ampere"] = pd.to_numeric(
    cargoterminal_el_demand["ampere"], errors="coerce"
)

## 3. Heavy Machineries schedules and consumptions

In [4]:
port_operations = pd.read_excel("inputs/Port_Operations.xlsx")
port_operations

,ID,Category,Model,Total working hours,Battery Capacity (kWh),Average Discharge Rate (kWh per hour),Working duration capacity (hours),Requires charging during operation,Fast Charge Capacity (kW)
0,Truck 417,Forklift,Merlinum EA 6-600,2,64,14.0,4.571429,no,128
1,Truck 405,Forklift,Merlinum EA 6-600,8,64,14.0,4.571429,yes,128
2,Truck 413,Forklift,Merlinum EA 6-600,4,64,14.0,4.571429,no,128
3,Truck 402,Forklift,Merlinum EA 6-600,2,64,14.0,4.571429,no,128
4,Truck 420,Forklift,SMV 16 Ever,11,110,20.0,5.500000,yes,147
5,Truck 422,Forklift,SMV 16 Ever,11,110,20.0,5.500000,yes,147
6,Truck 423,Forklift,SMV 16 Ever,11,110,20.0,5.500000,yes,147
7,Truck 424,Forklift,SMV 16 Ever,11,110,20.0,5.500000,yes,147
8,Truck 425,Forklift,SMV 16 Ever,11,110,20.0,5.500000,yes,147
9,Truck 426,Forklift,SMV 16 Ever,11,110,20.0,5.500000,yes,147


In [5]:
import numpy as np

dt = (
    pd.to_datetime(cargoterminal_el_demand["datetime"], errors="coerce")
    .dropna()
    .reset_index(drop=True)
)

# Machinery names from port_operations column "ID"
machinery_ids = (
    port_operations["ID"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

# Create HM_consumptions with datetime + one column per machinery, initialized to 0
HM_consumptions = pd.DataFrame({"datetime": dt})
HM_consumptions = pd.concat(
    [
        HM_consumptions,
        pd.DataFrame(
            np.zeros((len(HM_consumptions), len(machinery_ids)), dtype=float),
            columns=machinery_ids,
        ),
    ],
    axis=1,
)

# -------------------------------
# Random daily working schedules
# -------------------------------
# For each day and each vehicle:
# - choose random 15-minute work slots within allowed windows
# - add 1/4 of its average hourly discharge rate for each 15-minute slot
# - minimum consecutive run is 30 minutes (2 slots): isolated 15-minute slots are not allowed

rng = np.random.default_rng(42)  # change seed if you want a different random schedule

# Allowed working windows at 15-minute slot starts (minute offset from midnight)
ALLOWED_WINDOWS = [
    (7 * 60, 8 * 60 + 45),
    (10 * 60, 11 * 60 + 45),
    (13 * 60, 14 * 60 + 45),
    (16 * 60, 19 * 60 + 30),
]

ALLOWED_SLOTS = sorted(
    {
        m
        for start_m, end_m in ALLOWED_WINDOWS
        for m in range(start_m, end_m + 1, 15)
    }
)

WINDOW_SLOTS = [
    list(range(start_m, end_m + 1, 15))
    for start_m, end_m in ALLOWED_WINDOWS
]


def _slots_needed(total_hours: float) -> int:
    if pd.isna(total_hours):
        return 0
    try:
        h = float(total_hours)
    except Exception:
        return 0

    n = max(0, int(np.round(h * 4)))
    if n == 1:
        n = 2  # enforce minimum 30 minutes consecutive if any work is scheduled
    return min(n, len(ALLOWED_SLOTS))


def _window_feasible_counts(cap: int) -> list[int]:
    """Counts that can be represented as runs of at least 2 contiguous slots."""
    feasible = [0]
    for n in range(2, cap + 1):
        if n != 1:
            feasible.append(n)
    return feasible


WINDOW_CAPS = [len(ws) for ws in WINDOW_SLOTS]
WINDOW_FEASIBLE = [_window_feasible_counts(c) for c in WINDOW_CAPS]


def _allocate_slots_across_windows(total_slots: int, rng: np.random.Generator) -> list[int]:
    """Random feasible allocation of total slots across windows."""
    memo = {}

    def _can_allocate(i: int, rem: int) -> bool:
        key = (i, rem)
        if key in memo:
            return memo[key]
        if i == len(WINDOW_CAPS):
            memo[key] = (rem == 0)
            return memo[key]

        cap = WINDOW_CAPS[i]
        for x in WINDOW_FEASIBLE[i]:
            if x > cap or x > rem:
                continue
            if _can_allocate(i + 1, rem - x):
                memo[key] = True
                return True

        memo[key] = False
        return False

    if not _can_allocate(0, total_slots):
        raise RuntimeError("No feasible cross-window allocation for requested slots.")

    alloc = []
    rem = total_slots
    for i, cap in enumerate(WINDOW_CAPS):
        candidates = [x for x in WINDOW_FEASIBLE[i] if x <= cap and x <= rem and _can_allocate(i + 1, rem - x)]
        # Prefer using multiple windows (variegated timing), but keep randomization.
        if i < len(WINDOW_CAPS) - 1:
            weights = np.array([1.0 if x == 0 else 1.4 for x in candidates], dtype=float)
        else:
            weights = np.ones(len(candidates), dtype=float)
        weights = weights / weights.sum()

        x = int(rng.choice(np.array(candidates, dtype=int), p=weights))
        alloc.append(x)
        rem -= x

    return alloc


def _random_partition_runs(window_slots: int, rng: np.random.Generator) -> list[int]:
    """Random partition of window_slots into run lengths >=2."""
    if window_slots <= 0:
        return []
    if window_slots == 1:
        raise RuntimeError("Infeasible: single-slot run is not allowed.")

    runs = []
    rem = window_slots
    while rem > 0:
        if rem in (2, 3, 4):
            runs.append(rem)
            break

        opts = [2, 3, 4]
        valid = [o for o in opts if rem - o >= 0 and rem - o != 1]
        probs_map = {2: 0.45, 3: 0.35, 4: 0.20}
        probs = np.array([probs_map[o] for o in valid], dtype=float)
        probs = probs / probs.sum()

        choice = int(rng.choice(np.array(valid, dtype=int), p=probs))
        runs.append(choice)
        rem -= choice

    rng.shuffle(runs)
    return runs


def _build_window_pattern(cap: int, need: int, rng: np.random.Generator) -> np.ndarray:
    """Return random feasible mask of length cap with exactly need True slots and min run length 2."""
    from functools import lru_cache

    if need < 0 or need > cap or need == 1:
        raise RuntimeError("Infeasible single-window requirement.")

    @lru_cache(maxsize=None)
    def _count(pos: int, rem: int, prev_on: int, run_len: int) -> int:
        # End condition: no unfinished 1-slot run allowed.
        if pos == cap:
            if rem != 0:
                return 0
            if prev_on == 1 and run_len == 1:
                return 0
            return 1

        total = 0

        # Option 1: put 0 here (only if this does not end a 1-slot run)
        if not (prev_on == 1 and run_len == 1):
            total += _count(pos + 1, rem, 0, 0)

        # Option 2: put 1 here
        if rem > 0:
            next_run_len = run_len + 1 if prev_on == 1 else 1
            total += _count(pos + 1, rem - 1, 1, next_run_len)

        return total

    total_paths = _count(0, need, 0, 0)
    if total_paths == 0:
        raise RuntimeError("No feasible pattern exists for this window request.")

    out = np.zeros(cap, dtype=bool)
    pos, rem, prev_on, run_len = 0, need, 0, 0

    while pos < cap:
        count0 = 0
        if not (prev_on == 1 and run_len == 1):
            count0 = _count(pos + 1, rem, 0, 0)

        count1 = 0
        if rem > 0:
            next_run_len = run_len + 1 if prev_on == 1 else 1
            count1 = _count(pos + 1, rem - 1, 1, next_run_len)

        if count0 == 0 and count1 == 0:
            raise RuntimeError("Pattern construction failed unexpectedly.")

        if count0 == 0:
            take_one = True
        elif count1 == 0:
            take_one = False
        else:
            # Randomly sample among feasible continuations.
            p1 = count1 / (count0 + count1)
            take_one = bool(rng.random() < p1)

        if take_one:
            out[pos] = True
            rem -= 1
            run_len = run_len + 1 if prev_on == 1 else 1
            prev_on = 1
        else:
            out[pos] = False
            prev_on = 0
            run_len = 0

        pos += 1

    return out


def _pick_work_slots(total_hours: float, rng: np.random.Generator) -> list[int]:
    n = _slots_needed(total_hours)
    if n <= 0:
        return []

    allocations = _allocate_slots_across_windows(n, rng)
    selected = []

    for w_idx, need in enumerate(allocations):
        if need <= 0:
            continue

        ws = WINDOW_SLOTS[w_idx]
        cap = len(ws)
        pattern = _build_window_pattern(cap, need, rng)

        for i, take in enumerate(pattern):
            if take:
                selected.append(ws[i])

    if len(selected) != n:
        raise RuntimeError("Internal scheduler error: selected slot count mismatch.")

    return sorted(selected)


# Normalize datetimes and index by datetime for fast writes
hm = HM_consumptions.copy()
hm["datetime"] = pd.to_datetime(hm["datetime"], errors="coerce")
hm = hm.dropna(subset=["datetime"]).set_index("datetime").sort_index()

# Ensure energy columns exist for all IDs in port_operations
for vid in machinery_ids:
    if vid not in hm.columns:
        hm[vid] = 0.0

# Vehicle parameters from port_operations
port_ops = port_operations.copy()
port_ops["ID"] = port_ops["ID"].astype(str)
port_ops["Total working hours"] = pd.to_numeric(port_ops["Total working hours"], errors="coerce")
port_ops["Average Discharge Rate (kWh per hour)"] = pd.to_numeric(
    port_ops["Average Discharge Rate (kWh per hour)"], errors="coerce"
)

vehicle_params = (
    port_ops[["ID", "Total working hours", "Average Discharge Rate (kWh per hour)"]]
    .dropna(subset=["ID"])
    .drop_duplicates(subset=["ID"], keep="first")
)

# Reproduce behaviour for all days in the datetime series
all_days = hm.index.normalize().unique()

for day in all_days:
    for _, row in vehicle_params.iterrows():
        vid = row["ID"]
        total_hours = row["Total working hours"]
        rate_kwh_per_h = row["Average Discharge Rate (kWh per hour)"]

        if pd.isna(rate_kwh_per_h) or rate_kwh_per_h <= 0:
            continue

        for minute_offset in _pick_work_slots(total_hours, rng):
            ts = day + pd.Timedelta(minutes=minute_offset)
            if ts in hm.index:
                hm.loc[ts, vid] = float(hm.loc[ts, vid]) + float(rate_kwh_per_h) * 0.25

# Write back
HM_consumptions = hm.reset_index()

In [6]:
import os

# Choose output paths
output_paths = [
    "FerryTerminal_coding/inputs/HM_consumptions_15minutes.xlsx",
    "CargoTerminal_coding/inputs/HM_consumptions_15minutes.xlsx",
]

# Make a safe copy and ensure datetime format
hm_export = HM_consumptions.copy()
hm_export["datetime"] = pd.to_datetime(hm_export["datetime"], errors="coerce")

# Write file to both destinations
for output_path in output_paths:
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    hm_export.to_excel(output_path, index=False)

In [7]:
# -------------------------------
# Verification: daily energy check
# -------------------------------
# Verify for each day + vehicle:
#   sum(15-min kWh in HM_consumptions) == (scheduled 15-min slots / 4) * (Average Discharge Rate)
# using the same conversion/capping rule as the scheduler.

hm_check = HM_consumptions.copy()
hm_check["datetime"] = pd.to_datetime(hm_check["datetime"], errors="coerce")
hm_check = hm_check.dropna(subset=["datetime"]).set_index("datetime").sort_index()

# daily actual kWh per vehicle
actual_daily = hm_check.groupby(hm_check.index.normalize()).sum(numeric_only=True)

# expected kWh per vehicle (same for every day, based on port_operations)
expected_kwh_per_day = {}
for _, row in vehicle_params.iterrows():
    vid = row["ID"]
    rate = row["Average Discharge Rate (kWh per hour)"]
    slots = _slots_needed(row["Total working hours"])
    if pd.isna(rate) or rate <= 0 or slots <= 0:
        expected_kwh_per_day[vid] = 0.0
    else:
        expected_kwh_per_day[vid] = float(rate) * (float(slots) / 4.0)

mismatches = []

for day, day_row in actual_daily.iterrows():
    for vid, expected in expected_kwh_per_day.items():
        if vid not in actual_daily.columns:
            continue
        actual = float(day_row.get(vid, 0.0))
        if not np.isclose(actual, expected, rtol=0, atol=1e-9):
            mismatches.append((day, vid, expected, actual, actual - expected))

print(f"Checked days: {len(actual_daily)} | vehicles: {len(expected_kwh_per_day)}")
print(f"Mismatches: {len(mismatches)}")

if mismatches:
    mismatches_df = pd.DataFrame(
        mismatches,
        columns=["day", "vehicle", "expected_kWh", "actual_kWh", "diff_kWh"],
    ).sort_values(["day", "vehicle"])
    display(mismatches_df.head(50))
else:
    print("All daily totals match expected Total working hours × Average Discharge Rate.")


Checked days: 365 | vehicles: 32
Mismatches: 0
All daily totals match expected Total working hours × Average Discharge Rate.
